In [1]:
import os
import json
import pickle
import faiss
import numpy as np
from dotenv import load_dotenv

load_dotenv()
# Project Root & Processed Data Paths
PROJECT_ROOT = os.getcwd()
DATA_PROCESSED_DIR = os.path.join(PROJECT_ROOT, "data", "processed")

PATHS = {
    "chunks": os.path.join(DATA_PROCESSED_DIR, "chunks.json"),
    "faiss_index": os.path.join(DATA_PROCESSED_DIR, "faiss_index.bin"),
    "bm25_index": os.path.join(DATA_PROCESSED_DIR, "bm25_index.pkl")
}

# Load Master Metadata Registry & Indices
with open(PATHS["chunks"], "r", encoding="utf-8") as f:
    chunks = json.load(f)

faiss_index = faiss.read_index(PATHS["faiss_index"])

with open(PATHS["bm25_index"], "rb") as f:
    bm25_index = pickle.load(f)

print(f"Total Chunks in Registry : {len(chunks)}")
print(f"FAISS Index Vectors      : {faiss_index.ntotal}")
print(f"BM25 Corpus Size         : {bm25_index.corpus_size}")

Total Chunks in Registry : 2353
FAISS Index Vectors      : 2353
BM25 Corpus Size         : 2353


In [2]:
import torch
from sentence_transformers import SentenceTransformer, CrossEncoder

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

BI_ENCODER_NAME = "sentence-transformers/all-MiniLM-L6-v2"
CROSS_ENCODER_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"

print(f"Loading Bi-Encoder ({BI_ENCODER_NAME}) on {DEVICE}...")
bi_encoder = SentenceTransformer(BI_ENCODER_NAME, device=DEVICE)

print(f"Loading Cross-Encoder ({CROSS_ENCODER_NAME}) on {DEVICE}...")
cross_encoder = CrossEncoder(CROSS_ENCODER_NAME, max_length=512, device=DEVICE)


Loading Bi-Encoder (sentence-transformers/all-MiniLM-L6-v2) on cuda...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading Cross-Encoder (cross-encoder/ms-marco-MiniLM-L-6-v2) on cuda...


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

In [3]:
def hybrid_retrieve_and_rerank(query, candidate_k=20, final_top_k=5, rrf_k=60):
    """
    Stage 1: Parallel Dense (FAISS) + Sparse (BM25) Top-20 Retrieval
    Stage 2: Reciprocal Rank Fusion (RRF) Candidate Merging
    Stage 3: Cross-Encoder Joint Attention Re-Ranking to Top-5
    """
    # 1. Dense Retrieval (FAISS)
    query_embedding = bi_encoder.encode([query], normalize_embeddings=True).astype("float32")
    dense_scores, dense_indices = faiss_index.search(query_embedding, candidate_k)
    dense_ranked_ids = dense_indices[0].tolist()

    # 2. Sparse Retrieval (BM25)
    tokenized_query = query.lower().split()
    bm25_scores = bm25_index.get_scores(tokenized_query)
    sparse_ranked_ids = np.argsort(bm25_scores)[::-1][:candidate_k].tolist()

    # 3. Reciprocal Rank Fusion (RRF)
    rrf_scores = {}
    for rank, doc_idx in enumerate(dense_ranked_ids):
        rrf_scores[doc_idx] = rrf_scores.get(doc_idx, 0.0) + (1.0 / (rrf_k + rank + 1))

    for rank, doc_idx in enumerate(sparse_ranked_ids):
        rrf_scores[doc_idx] = rrf_scores.get(doc_idx, 0.0) + (1.0 / (rrf_k + rank + 1))

    # Select Top-20 fused candidates
    fused_ids = sorted(rrf_scores.keys(), key=lambda idx: rrf_scores[idx], reverse=True)[:candidate_k]

    # 4. Cross-Encoder Re-Ranking
    candidate_pairs = [(query, chunks[idx]["chunk_text"]) for idx in fused_ids]
    cross_scores = cross_encoder.predict(candidate_pairs)

    # Combine & sort by Cross-Encoder score descending
    reranked_results = []
    for idx, ce_score in zip(fused_ids, cross_scores):
        chunk_data = chunks[idx].copy()
        chunk_data["rrf_score"] = round(float(rrf_scores[idx]), 4)
        chunk_data["cross_encoder_score"] = round(float(ce_score), 4)
        reranked_results.append(chunk_data)

    reranked_results.sort(key=lambda x: x["cross_encoder_score"], reverse=True)
    return reranked_results[:final_top_k]



In [4]:
import time

test_queries = [
    "Who is the Director of IIIT Nagpur and what is his contact number?",
    "CSL-422 Machine Learning syllabus or faculty",
    "Hostel rules and fee structure details"
]

for q in test_queries:
    start_time = time.time()
    results = hybrid_retrieve_and_rerank(q, candidate_k=20, final_top_k=5)
    latency_ms = (time.time() - start_time) * 1000

    print("=" * 80)
    print(f"QUERY: {q}  |  Latency: {latency_ms:.1f} ms")
    print("=" * 80)

    for rank, res in enumerate(results, start=1):
        # Strip the prepended metadata header for clean preview display
        body_preview = res["chunk_text"].split("\n\n", 1)[-1][:180].replace("\n", " ")
        print(f"Rank {rank} | CE Score: {res['cross_encoder_score']} | RRF: {res['rrf_score']}")
        print(f"   Title  : {res['title']} ({res['category']})")
        print(f"   Source : {res['source_url']}")
        print(f"   Preview: {body_preview}...\n")

QUERY: Who is the Director of IIIT Nagpur and what is his contact number?  |  Latency: 17531.2 ms
Rank 1 | CE Score: 5.2963 | RRF: 0.0161
   Title  : Indian Institute of Information Technology Nagpur (Governance)
   Source : https://www.iiitn.ac.in/governance/administration
   Preview: the software i... More about Chairman director Dr. Prem Lal Patel director 📧 Email: director@iiitn.ac.in Professor Prem Lal Patel, Director, VNIT, Nagpur took over the additional c...

Rank 2 | CE Score: 3.5027 | RRF: 0.0143
   Title  : View English PDF (Institute)
   Source : https://iiitn.ac.in/media/818b135d91ef496ca2c647dec6df5e6a.pdf
   Preview: Excellence (CoE) at IIIT, Nagpur. Director of IIIT, Nagpur Dr Omprakash Kakde said the faculty and students of the institute will contribute to the working of CoE. The MoU was sign...

Rank 3 | CE Score: 3.4296 | RRF: 0.0154
   Title  : Read More → (General)
   Source : https://iiitn.ac.in/media/1ba7243672ec400aa6a1ffbc648cadf1.pdf
   Preview: --- Page 1 ---

In [6]:
import os
from typing import List
from langchain_core.documents import Document
from langchain_core.retrievers import BaseRetriever
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain_ollama import ChatOllama

class CampusNLPHybridRetriever(BaseRetriever):
    candidate_k: int = 20
    final_top_k: int = 5
    ce_threshold: float = -1.0

    def _get_relevant_documents(self, query: str) -> List[Document]:
        raw_results = hybrid_retrieve_and_rerank(
            query=query,
            candidate_k=self.candidate_k,
            final_top_k=self.final_top_k
        )

        # Filter negative-logit boilerplate noise while always keeping Rank 1
        filtered = [
            r for i, r in enumerate(raw_results)
            if i == 0 or r["cross_encoder_score"] >= self.ce_threshold
        ]

        docs = []
        for idx, r in enumerate(filtered, start=1):
            docs.append(
                Document(
                    page_content=r["chunk_text"],
                    metadata={
                        "source_num": idx,
                        "chunk_id": r["chunk_id"],
                        "title": r["title"],
                        "category": r["category"],
                        "source_url": r["source_url"],
                        "rrf_score": r["rrf_score"],
                        "cross_encoder_score": r["cross_encoder_score"]
                    }
                )
            )
        return docs


# 3. Initialize Retriever & Local Qwen 2.5 via ChatOllama
retriever = CampusNLPHybridRetriever(candidate_k=20, final_top_k=5, ce_threshold=-1.0)

llm = ChatOllama(
    model="qwen2.5:latest",
    temperature=0.1,
    num_ctx=4096
)

# 4. Grounded Prompt & LCEL Chain with Source Attribution
prompt = ChatPromptTemplate.from_messages([
    ("system", """You are CampusNLP, an official AI assistant for Indian Institute of Information Technology (IIIT) Nagpur.
Answer the user's question strictly using ONLY the provided context passages below.
- Do NOT invent, assume, or extrapolate any facts not explicitly present in the context.
- If the context does not contain the answer, state clearly: "The provided IIIT Nagpur documents do not contain enough information to answer this question."
- Cite the relevant source numbers (e.g., [Source 1]) inline where appropriate.
- Present structured data (such as fees, schedules, or course codes) cleanly using bullet points or tables."""),
    ("human", "CONTEXT:\n{context}\n\nUSER QUESTION: {question}\n\nGROUNDED ANSWER:")
])


def format_docs_with_sources(docs: List[Document]) -> str:
    blocks = []
    for doc in docs:
        src_num = doc.metadata["source_num"]
        blocks.append(f"[Source {src_num}]\n{doc.page_content}")
    return "\n\n---\n\n".join(blocks)


rag_chain = (
    RunnableParallel(
        docs=retriever,
        question=RunnablePassthrough()
    )
    .assign(
        answer=(
                RunnablePassthrough.assign(context=lambda x: format_docs_with_sources(x["docs"]))
                | prompt
                | llm
                | StrOutputParser()
        )
    )
)

# 5. Run Test Queries & Log Traces to LangSmith
sample_queries = [
    "Who is the Director of IIIT Nagpur and what is his email?",
    "What are the credits and L-T-P structure for CSL-422 Machine Learning?",
    "What is the hostel fee structure for 2026-27?"
]

for q in sample_queries:
    result = rag_chain.invoke(q)
    print("=" * 80)
    print(f"QUESTION: {result['question']}\n")
    print(result["answer"])
    print("\nVerified Sources:")
    seen_urls = set()
    for d in result["docs"]:
        url = d.metadata["source_url"]
        if url not in seen_urls:
            print(
                f" - [Source {d.metadata['source_num']}] {d.metadata['title']} (CE: {d.metadata['cross_encoder_score']}): {url}")
            seen_urls.add(url)
    print("=" * 80 + "\n")

QUESTION: Who is the Director of IIIT Nagpur and what is his email?

The Director of IIIT Nagpur mentioned in the document is Prof. Dr. Prem Lal Patel. However, the specific email address for him or any other official email for the director position at IIIT Nagpur is not provided in the given text. For accurate and up-to-date contact information, you should visit the official IIIT Nagpur website or contact their main office directly.

Verified Sources:
 - [Source 1] Indian Institute of Information Technology Nagpur (CE: 7.1122): https://www.iiitn.ac.in/governance/administration
 - [Source 2] Read More → (CE: 4.9713): https://iiitn.ac.in/media/1ba7243672ec400aa6a1ffbc648cadf1.pdf
 - [Source 3] View English PDF (CE: 4.4415): https://iiitn.ac.in/media/818b135d91ef496ca2c647dec6df5e6a.pdf
 - [Source 4] Read More → (CE: 3.8589): https://iiitn.ac.in/media/4408aa0356994798827a56612b398256.pdf

QUESTION: What are the credits and L-T-P structure for CSL-422 Machine Learning?

The credits and L-